# 2.0版本控制台

In [ ]:
from backend2.l1.pipeline import run_l1_pipeline
import time

configs = [
    {
        "dataset_id": "h5_full",
        "reader": {
            "kind": "h5",
            "path": "datasets/2D_rdb_NA_NA.h5",
            "dataset": "data",
            "fill_value": 0.0,
        },
        "split": {
            "strategy": "temporal",
            "unit": "frame",
            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
            "seed": 123,
        },
        "normalization": {"method": "zscore"},
        "artifacts_dir": "artifacts",
        "save_array5d": False,
        "log_progress": True,
    },
    {
        "dataset_id": "nc_full",
        "reader": {
            "kind": "nc",
            "path": "datasets/cylinder2d.nc",
            "var_keys": ["u", "v"],
            "time_key": "tdim",
            "y_key": "ydim",
            "x_key": "xdim",
            "fill_value": 0.0,
        },
        "split": {
            "strategy": "temporal",
            "unit": "frame",
            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
            "seed": 123,
        },
        "normalization": {"method": "zscore"},
        "artifacts_dir": "artifacts",
        "save_array5d": False,
        "log_progress": True,
    },
    {
        "dataset_id": "sst_full",
        "reader": {
            "kind": "mat",
            "path": "datasets/sst_weekly.mat",
            "var": "sst",
            "lon_key": "lon",
            "lat_key": "lat",
            "time_key": "time",
            "fill_value": 0.0,
        },
        "split": {
            "strategy": "temporal",
            "unit": "frame",
            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
            "seed": 123,
        },
        "normalization": {"method": "zscore"},
        "artifacts_dir": "artifacts",
        "save_array5d": False,
        "log_progress": True,
    },
]

summaries = []
total = len(configs)
overall_t0 = time.perf_counter()

for i, cfg in enumerate(configs, start=1):
    print(f"\n[PIPELINE] ({i}/{total}) start: {cfg['dataset_id']}", flush=True)
    t0 = time.perf_counter()
    summary = run_l1_pipeline(cfg)
    dt = time.perf_counter() - t0
    summaries.append(
        {
            "dataset_id": summary.dataset_id,
            "shape5d": list(summary.shape5d),
            "split_sizes": summary.split_sizes,
            "stats_method": summary.stats_method,
            "artifacts_dir": summary.artifacts_dir,
            "elapsed_sec": round(dt, 2),
        }
    )
    print(f"[PIPELINE] ({i}/{total}) done: {summary.dataset_id} in {dt:.2f}s", flush=True)

print(f"\n[PIPELINE] all done in {time.perf_counter() - overall_t0:.2f}s", flush=True)
summaries

In [ ]:
from backend2.l2.train import run_l2_train
from backend2.l2.infer import run_l2_infer
from backend2.l2.utils import now_tag
import json

datasets = ["h5_full", "nc_full", "sst_full"]
exp_name = "baseline_unet"
split_tag = "temporal_frame_seed123"

all_summaries = []

for dataset_id in datasets:
    run_name = f"run_{dataset_id}_{now_tag()}"
    train_cfg = {
        "dataset_id": dataset_id,
        "artifacts_dir": "artifacts",
        "exp_name": exp_name,
        "run_name": run_name,
        "device": "auto",
        "seed": 123,
        "split_tag": split_tag,
        "target_offset": 1,
        "batch_size": 8,
        "num_workers": 0,
        "epochs": 20,
        "lr": 1e-3,
        "model": {
            "base_channels": 32,
            "convs_per_stage": 2,
        },
    }
    infer_cfg = {
        "dataset_id": dataset_id,
        "artifacts_dir": "artifacts",
        "exp_name": exp_name,
        "run_name": run_name,
        "device": "auto",
        "split_tag": split_tag,
        "target_offset": 1,
        "batch_size": 8,
        "num_workers": 0,
        "ckpt_name": "model_best.pt",
        "model": {
            "base_channels": 32,
            "convs_per_stage": 2,
        },
        "probe": {
            "enabled": True,
            "record_level": 0,
            "hook_layers": ["enc.stage*.out", "dec.stage*.out", "skip.*", "head.out"],
        },
    }

    train_summary = run_l2_train(train_cfg)
    infer_summary = run_l2_infer(infer_cfg)

    all_summaries.append(
        {
            "dataset_id": dataset_id,
            "run_name": run_name,
            "train": train_summary,
            "infer": infer_summary,
        }
    )

print(json.dumps(all_summaries, ensure_ascii=False, indent=2))